In [ ]:
# ---------------- Imports ----------------
import os
import json
from collections import Counter
from collections import defaultdict

import yaml
import pandas as pd
from transformers import AutoTokenizer
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl
import os

# Path to font
FONT_DIR = os.path.join("../../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,
})


In [ ]:
# ---------------- Args ----------------

RESULT_FILES = {
    "Llama-8B": "meta-llama/Llama-3.1-8B-Instruct/llama-3.1-8b-authoritative-alpha-var-plot-data.csv",
    "Mistral-7B": "mistralai/Mistral-7B-Instruct-v0.3/mistral-7b-instruct-consensus-alpha-var-plot-data.csv",
}

OUTPUT_FILE = "multi-model-alpha-plot.pdf"


In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]


RESULTS_DIR = os.path.join(PROJ_STORE, "evaluation", "alpha-plot")

# OUTPUT
OUTPUT_DIR = os.path.join(PROJ_STORE, "evaluation", "alpha-plot-multmodel")
os.makedirs(OUTPUT_DIR, exist_ok=True)



# ---------------- Build filename prefix ----------------
model_prefix = "-".join(
    sorted(name.lower().replace(" ", "").replace("/", "-") 
           for name in RESULT_FILES.keys())
)

prefixed_output_file = f"{model_prefix}-{OUTPUT_FILE}"


In [ ]:
# %%
# ---------------- Plot ----------------

plt.figure(figsize=(5, 5))

# Color per model (NeurIPS-safe contrast)
MODEL_COLORS = {
    "Llama-8B": "#004C80",
    "Mistral-7B": "#4A4A4A",
}

        
for model_name, relative_path in RESULT_FILES.items():

    full_path = os.path.join(RESULTS_DIR, relative_path)

    if not os.path.exists(full_path):
        raise FileNotFoundError(f"Missing: {full_path}")

    df = pd.read_csv(full_path)
    df = df.sort_values("alpha", ascending=False)

    alphas = df["alpha"]

    # MSPR (solid line)
    plt.errorbar(
        alphas,
        df["MSPR_mean"],
        yerr=df["MSPR_std"],
        marker="o",
        linestyle="-",
        linewidth=2,
        capsize=6,
        color=MODEL_COLORS.get(model_name, None),
        label=f"{model_name} MSPR"
    )

    # Balanced Accuracy (dashed line)
    plt.errorbar(
        alphas,
        df["BAL_mean"],
        yerr=df["BAL_std"],
        marker="s",
        linestyle="--",
        linewidth=2,
        capsize=6,
        color=MODEL_COLORS.get(model_name, None),
        label=f"{model_name} BAcc"
    )

# Reverse alpha axis (1 → 0)
plt.gca().invert_xaxis()

plt.xlabel("Alpha (Loss Weight)", fontsize=22)
plt.ylabel("Percentage (%)", fontsize=22)

plt.xticks(fontsize=22)
plt.yticks(fontsize=22)

plt.legend(
    fontsize=16, 
    loc="upper left",
    bbox_to_anchor=(0.0, 1),  # (x, y) in axes coordinates
    borderaxespad=0.0,
    framealpha=0.3   # 0 = fully transparent, 1 = fully opaque
)
plt.tight_layout()

final_output_path = os.path.join(OUTPUT_DIR, prefixed_output_file)
plt.savefig(final_output_path, format="pdf", bbox_inches="tight")
print(f"Saved to: {final_output_path}")
plt.show()
